# B2 책 특징 벡터 확인

A3의 `load_book_tags()`와 B1의 `filter_book_tags()` 결과로 책 x 태그 행렬을 만듭니다.
태그 이름 하나를 특징 하나로 사용합니다. 원본 데이터와 다른 팀원의 코드는 수정하지 않습니다.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from bookrec.data import load_book_tags, load_books
from bookrec.data.tag_filter import filter_book_tags
from bookrec.models.base import BOOK_IDS
from bookrec.models.content_based.features import build_book_features

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
OUT = ROOT / "results" / "content_based"
OUT.mkdir(parents=True, exist_ok=True)
books = load_books()
tags = filter_book_tags(load_book_tags())
print(f"B1 정제 결과: {len(tags):,}행, {tags['book_id'].nunique():,}권")


B1 정제 결과: 700,957행, 10,000권


## 1. 특징 행렬 만들기

기본 설정은 TF-IDF, 최소 등장 책 수 2, sublinear TF입니다.
한 권에만 붙은 태그를 줄이고, 큰 count는 `1 + log(count)`로 완화합니다.
IDF는 `log((1 + 전체 책 수) / (1 + 등장 책 수)) + 1`이며 마지막에 책별 L2 정규화를 적용합니다.
이 설정은 초기 후보이며, B4에서 검증 데이터로 최종 설정을 고릅니다.


In [2]:
features = build_book_features(tags)
matrix = features.matrix
lengths = np.sqrt(matrix.multiply(matrix).sum(axis=1)).A1
nonzero = lengths > 0
summary = pd.DataFrame([{
    "weighting": "tfidf", "min_df": 2, "sublinear_tf": True,
    "books": matrix.shape[0], "features": matrix.shape[1], "nonzero_values": matrix.nnz,
    "density": matrix.nnz / (matrix.shape[0] * matrix.shape[1]),
    "zero_books": int((~nonzero).sum()),
    "csr_bytes": matrix.data.nbytes + matrix.indices.nbytes + matrix.indptr.nbytes,
}])
print(summary.to_string(index=False))
summary.to_csv(OUT / "B2_feature_summary.csv", index=False)


weighting  min_df  sublinear_tf  books  features  nonzero_values  density  zero_books  csr_bytes
    tfidf       2          True  10000     16958          683823 0.004032           0    8245880


## 2. 공통 인터페이스와 수치 확인

책의 행 순서는 `BOOK_IDS`(1~10,000)와 같습니다. 태그가 없는 책도 영벡터로 남깁니다.
0이 아닌 책 벡터 길이는 1이며, 입력 count와 생성한 특징에는 유한한 값만 허용합니다.


In [3]:
checks = pd.DataFrame([
    ("책 행 순서가 BOOK_IDS와 같음", np.array_equal(features.book_ids, BOOK_IDS)),
    ("태그 열 이름에 중복 없음", len(np.unique(features.tag_names)) == matrix.shape[1]),
    ("희소 행렬 값이 모두 유한함", np.isfinite(matrix.data).all()),
    ("특징 가중치가 음수가 아님", (matrix.data >= 0).all()),
    ("0이 아닌 벡터의 L2 길이가 1", np.allclose(lengths[nonzero], 1)),
], columns=["check", "passed"])
print(checks.to_string(index=False))
assert checks["passed"].all()


               check  passed
책 행 순서가 BOOK_IDS와 같음    True
      태그 열 이름에 중복 없음    True
     희소 행렬 값이 모두 유한함    True
      특징 가중치가 음수가 아님    True
  0이 아닌 벡터의 L2 길이가 1    True


## 3. 비슷한 책 확인

정규화된 벡터의 내적으로 코사인 유사도를 구합니다. 자기 자신과 유사도가 0인 책은 제외하고,
동점이면 작은 책 ID를 먼저 둡니다. 대표 책 5권마다 상위 5권을 확인합니다.
제목의 유사성은 정성적 확인일 뿐이며, 추천 성능은 B3~B4의 독자별 평가로 판단합니다.


In [4]:
titles = books.set_index("book_id")["title"]
examples = []
for book_id in [1, 2, 3, 100, 10000]:
    similar = features.similar_books(book_id, k=5)
    similar.insert(0, "source_book_id", book_id)
    similar.insert(1, "source_title", titles.loc[book_id])
    similar["title"] = similar["book_id"].map(titles)
    examples.append(similar)
examples = pd.concat(examples, ignore_index=True)
print(examples.to_string(index=False))
examples.to_csv(OUT / "B2_similar_books.csv", index=False)


 source_book_id                                             source_title  book_id  similarity                                                                                     title
              1                  The Hunger Games (The Hunger Games, #1)       17    0.934240                                                      Catching Fire (The Hunger Games, #2)
              1                  The Hunger Games (The Hunger Games, #1)       20    0.872879                                                         Mockingjay (The Hunger Games, #3)
              1                  The Hunger Games (The Hunger Games, #1)      507    0.829800                                  The Hunger Games Trilogy Boxset (The Hunger Games, #1-3)
              1                  The Hunger Games (The Hunger Games, #1)       12    0.640206                                                                 Divergent (Divergent, #1)
              1                  The Hunger Games (The Hunger Games, #1)      32

## 다음 작업

B3에서는 `features.matrix`로 독자 취향 벡터와 전체 책의 점수를 계산하고,
B5에서는 같은 특징 공간에서 k-means를 수행합니다.
count/binary 방식도 같은 함수의 `weighting`으로 선택할 수 있습니다.
전 책 간 유사도 행렬이나 큰 특징 행렬 파일은 저장하지 않고 입력과 설정으로 재생성합니다.
